# Discovery of the Higgs Boson

Final Project, Probability and Statistics for Data Analysis

On July 4, 2012, the ATLAS and CMS experiments at the LHC announced a new
particle consistent with the Standard Model Higgs boson. This notebook hands you
real CMS four-lepton events from that era, about half the data the four-lepton
channel had at the announcement, plus the tools of this course: a classifier as
a likelihood-ratio approximation, a generalized likelihood-ratio test with a
nuisance parameter, toy p-values, goodness-of-fit, and a power calculation. The
question the project answers is the one the experiments faced: what are you
entitled to claim from this data?

### <font color="#A6631F">Predict</font>
<hr color="#A6631F">

Before running anything, write down what you expect. Will half the
announcement-era data in a single channel show the particle? At what
significance, if any?

Two through-lines from Module 5 carry over, and both get sharper teeth here.
First, significant is not adequate: a test verdict is only as good as the model
it tests against. Second, rejecting the null is not confirming your alternative.

The work runs in two threads. Sections 2 and 3 turn a classifier into a test
statistic. Sections 4 through 7 set the classifier aside and run the discovery
test itself on the mass spectrum, then ask how solid it is and what more data
would buy. Section 8 returns to the classifiers and measures what the analysis
gains from them. Section 9 settles the opening question, and its checkpoint is
graded before you submit.

Run with the seed set below. You write the statistical core yourself: the
Poisson likelihood, the fit that produces the test statistic, the toy
calibration, the goodness-of-fit test, and the confidence interval. The modules
`util.py` and `higgs_analysis.py` beside this notebook carry the classifier
plumbing and a tested reference implementation, which the checkpoint cells use
to verify what you wrote. The scientific result (Sections 4 to 6) does not
depend on which classifier you build.


## What is graded

Seven tasks, four gates.

- **Readiness (R1 to R3).** Tasks 1 through 5 are the statistical core. Each one
  ends in a checkpoint cell that runs your code on fixed inputs and compares it
  against the tested reference in `higgs_analysis.py`. Clear every checkpoint
  before you go on; later sections run on what you built.
- **Experiment record (R4 to R8).** The values your pipeline produces, entered
  from the Section 10 cell.
- **The comparison gate.** Task 6 asks you to make a model comparison fair
  before you read it. The graded checkpoint follows Section 8.
- **The inference gate.** Task 7 is your written claim, evidence, caveat, and
  next experiment. The graded checkpoint follows Section 9.

Two conventions mark the work, and they mean different things:

- A stub ending in `raise NotImplementedError` is a function to write from the
  formula stated above it.
- A `...` on the right of an assignment is one line to fill in, inside
  scaffolding that is already correct.

Cells labeled `# Provided:` are yours to read and run, not to edit. They handle
data loading, plotting, and the classifier fitting that is not the subject of
this course.

### <font color="#8A4E17">Watch out</font>
<hr color="#8A4E17">

The reference implementation sits in <code>higgs_analysis.py</code>, and the checkpoints call it to check your answers. Calling it from inside your own functions defeats the point of the task and will pass the checkpoint while teaching you nothing. Write the arithmetic yourself; use the reference only where a cell already does.

## 0. Setup: run this cell first

If you opened this notebook standalone (for example from the Colab badge), the
helper modules and data files are not in your session yet. This cell fetches
them from the project repository. Run locally from a full checkout and it does
nothing.


In [ ]:
# Colab bootstrap: fetch helper modules and data if they aren't already here.
import os, urllib.request

REF = "main"   # pin to a release tag or commit SHA before the offering opens
REPO = f"https://raw.githubusercontent.com/codey-m/prob_stats/{REF}/final_project"
NEEDED = [
    "util.py", "higgs_analysis.py",
    "data/MC/higgs2012.csv", "data/MC/zzto4mu2012.csv",
    "data/MC/zzto2mu2e2012.csv", "data/MC/zzto4e2012.csv",
    "data/data/clean_data_2012.csv",
]
for rel in NEEDED:
    if not os.path.exists(rel):
        folder = os.path.dirname(rel)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(f"{REPO}/{rel}", rel)
        print(f"downloaded {rel}")
print("environment ready")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm, chi2

import util
import higgs_analysis as H

# SEED fixes the toys, the MLP's initialization, and (because sklearn's MLP
# takes no sample weights) which rows are resampled into its training set.
# The train/holdout split is pinned separately at seed 0 in H.prepare below,
# so the holdout is identical across runs and across students.
SEED = 6372
np.random.seed(SEED)
plt.rcParams["figure.figsize"] = (7, 4)


## 1. The data and the simulation

Two ingredients.

Experimental data: about 495 real CMS events that survived the H→ZZ→4ℓ
selection. Each event is four leptons (electrons or muons), described by their
four-momenta $(E, p_x, p_y, p_z)$ plus reconstructed masses.

Monte Carlo simulation (MC): labeled events for the Higgs signal and for the
three ZZ→4ℓ background processes (`zz4mu`, `zz2mu2e`, `zz4e`), which produce
the same four-lepton signature as the signal.

Two further backgrounds, Drell-Yan and $t\bar t$, are not included. Their
combined yield (about 10 events) is comparable to the signal itself, but the
available simulation keeps only a handful of rows for them, too few to model a
mass shape, and resampling those rows would repeat information rather than add
it. Nothing downstream models them or corrects for them. Treat that as a stated
limitation on every number this project reports; Section 9 returns to it.

Each MC process carries a physical weight, the number of expected events one
simulated row represents. Summing weights gives a process's expected yield. The
next cell tallies them against the observed count.


In [ ]:
# Provided: load the data and the simulation, and report expected yields.
st = H.prepare(seed=0)
data = st["data"]
weights = util.compute_weights()

print(f"experimental events: {len(data)}")
print("MC process yields (expected events at this luminosity):")
for name, p, w in zip(util.PROCESS_NAMES, st["processes"], weights):
    print(f"  {name:9s}: {len(p):6d} rows  ->  yield {w*len(p):8.2f}")
tot_bkg = sum(w*len(p) for w, p in zip(weights[1:], st["processes"][1:]))
print(f"\ntotal expected background: {tot_bkg:.1f}")
print(f"total expected signal:     {weights[0]*len(st['processes'][0]):.1f}")
print(f"observed:                  {len(data)}   (data/prediction = "
      f"{len(data)/(tot_bkg+weights[0]*len(st['processes'][0])):.3f})")


### <font color="#8A4E17">What you found</font>
<hr color="#8A4E17">

The signal is tiny: about 9 expected Higgs events on 411 expected background.
And the data run 18 percent above the total prediction, so the simulation's
normalization is off. Section 4 has to absorb that mismatch before any test of
"is there a Higgs" can be trusted.


## 2. Weighting for the Neyman-Pearson connection

The most powerful test of background $P_0$ against signal $Q$ uses the
log-likelihood ratio $h^*(x) = \log\big(Q(x)/P_0(x)\big)$. That is the
Neyman-Pearson lemma. We do not know $P_0$ or $Q$ in closed form, only samples
from each, but a classifier trained with cross-entropy has log-odds that
converge to $h^*$ plus a constant $\log(\nu_1/\nu_0)$ set by the class
proportions. Giving both classes equal total weight makes the constant zero, so
the trained log-odds read as $h^*$ directly.

A classifier's last layer computes a real number $z(x)$ and passes it through
the sigmoid, $f = e^z/(1 + e^z)$. Then $1 - f = 1/(1 + e^z)$, so

$$\log\big(f/(1-f)\big) = \log\big(e^z\big) = z.$$

The log-odds is not something you compute from the classifier's output. It is
the number the classifier already had, one step before the sigmoid. That is why
`score()` reads `decision_function` where a model exposes it and recovers
`log(p/(1-p))` where it does not: the two are the same quantity.

The offset $\log(\nu_1/\nu_0)$ is not a formality. Train on the physical
weights instead, where the signal yield is 9.38 against 411.30 of background,
and the offset is $\log(9.38/411.30) = -3.78$. A threshold at $z = 0$ would
then sit at $h^* = 3.78$, a likelihood ratio near 44, not the even-odds cut it
looks like.

This balancing is a convenience, not a requirement. The constant is the same
for every event, so it moves every score by the same amount and changes no
ranking, no AUC, and none of the score bins built from quantiles in Section 8.
We balance so that log-odds are readable on an absolute scale and a fixed
threshold means the same thing across models.

`compute_weights()` returns the physical weights, and `class_balanced_weights()`
rescales them to equal class totals for training. The check below confirms the
balance, and then we train.


## Your task 1: Balance the two classes

The log-odds of a cross-entropy classifier converge to
$h^*(x) + \log(\nu_1/\nu_0)$, where $\nu_1$ and $\nu_0$ are the total weights of
the signal and background training samples. Make that offset vanish.

Write `class_balanced_weights(signal_mask, physical_weights)`. Rescale the
signal weights and the background weights separately so that each class sums to
1, leaving the relative weights inside a class untouched. Return a new array;
do not modify the input.

In [ ]:
# STUDENT TASK 1: equal total weight per class.

def class_balanced_weights(signal_mask, physical_weights):
    """Rescale so signal and background each carry total weight 1."""
    signal_mask = np.asarray(signal_mask, bool)
    w = np.asarray(physical_weights, float).copy()
    # TODO 1: divide the signal entries by their total, and the background
    #         entries by theirs, so each class sums to 1.
    raise NotImplementedError("Return the class-balanced weight array.")


In [ ]:
# Checkpoint 1: fixed inputs, then the real training sample.
_mask = np.array([True, True, False, False, False])
_phys = np.array([1.0, 3.0, 2.0, 2.0, 4.0])
_got = class_balanced_weights(_mask, _phys)
assert np.allclose(_got, [0.25, 0.75, 0.25, 0.25, 0.5]), \
    "Each class must sum to 1 with within-class proportions preserved."
assert np.allclose(_phys, [1.0, 3.0, 2.0, 2.0, 4.0]), "Do not modify the input array."

full = st["train"]
sig = (full["_proc"] == 0).values
bal = class_balanced_weights(sig, full["_w"].values)

# The reference in util.py normalizes to a different overall scale. Only the
# RATIO of the two class totals sets the log(nu1/nu0) offset, so agreement up to
# one positive constant is the property that matters.
_scale = util.class_balanced_weights(sig, full["_w"].values) / bal
assert np.allclose(_scale, _scale[0]), \
    "Your weights should match the reference up to a single overall factor."

print(f"total signal weight     : {bal[sig].sum():.1f}")
print(f"total background weight  : {bal[~sig].sum():.1f}")
print(f"ratio (should be 1.000)  : {bal[sig].sum()/bal[~sig].sum():.3f}")
print("\u2713 Checkpoint 1 passed.")


## 3. A test statistic from a classifier

We hand the classifier only per-lepton kinematics, 24 numbers per event
(energy, the three momentum components, and the angles `eta` and `phi`, for
each of the four leptons), and withhold the reconstructed masses. The model has to find
the structure itself. Start with logistic regression, the Unit 1 model.

### <font color="#A6631F">Predict</font>
<hr color="#A6631F">

A weighted AUC of 0.5 is chance and 1.0 is perfect separation. Where
does a linear model on these 24 numbers land? Write your number down.


In [ ]:
# Provided: fit the linear model and look at its scores.
lin, featfn = H.make_classifier("linear", st["train"])
auc_lin = H.auc(lin, featfn, st["holdout"])
print(f"linear logistic regression, weighted AUC = {auc_lin:.3f}")

# distributions of h(x) for signal MC, background MC, and real data
h_sig  = H.score(lin, featfn, st["processes"][0])
h_bkg  = np.concatenate([H.score(lin, featfn, p) for p in st["processes"][1:]])
h_data = H.score(lin, featfn, data)
bins = np.linspace(min(h_bkg.min(), h_data.min()), max(h_sig.max(), h_data.max()), 40)
plt.hist(h_bkg,  bins=bins, density=True, alpha=0.4, label="background MC")
plt.hist(h_sig,  bins=bins, density=True, alpha=0.4, label="signal MC")
plt.hist(h_data, bins=bins, density=True, alpha=0.4, label="data")
plt.xlabel(r"$h(x)$ (linear model log-odds)"); plt.ylabel("density"); plt.legend()
plt.title(f"Linear model barely separates signal from background (AUC {auc_lin:.2f})")
plt.show()


### <font color="#8A4E17">What you found</font>
<hr color="#8A4E17">

Barely better than chance. Is the Higgs simply not separable from the
ZZ background? No. The discriminating variable is the four-lepton invariant
mass, and there is a structural reason a linear model cannot see it:

$$ m^2 = \Big(\textstyle\sum_i E_i\Big)^2 - \Big(\sum_i p_{x,i}\Big)^2
        - \Big(\sum_i p_{y,i}\Big)^2 - \Big(\sum_i p_{z,i}\Big)^2 $$

The mass is a quadratic form in the raw inputs, and no linear function can
represent it. Don't take that on faith: the next cell checks the identity
numerically against the provided mass column.


In [ ]:
# Provided: rebuild the four-lepton mass from the raw inputs.
d = data
E  = d[["E1","E2","E3","E4"]].values.sum(1)
px = d[["px1","px2","px3","px4"]].values.sum(1)
py = d[["py1","py2","py3","py4"]].values.sum(1)
pz = d[["pz1","pz2","pz3","pz4"]].values.sum(1)
m_reconstructed = np.sqrt(np.maximum(E**2 - px**2 - py**2 - pz**2, 0))
print(f"max |reconstructed mass - provided mass| = "
      f"{np.abs(m_reconstructed - d['mass'].values).max():.2e} GeV")
print("=> the target is a quadratic form in the inputs; a linear model cannot represent it.")


So give the model the capacity to form products of inputs. Two routes: a
quadratic feature expansion on the raw inputs, which can represent $m^2$
exactly while staying inside Unit 1 machinery, or a small neural network, which
has to approximate it from the raw inputs on its own.


In [ ]:
# Provided: two models with the capacity to form products of inputs.
quad, _ = H.make_classifier("quadratic", st["train"])
mlp,  _ = H.make_classifier("mlp", st["train"])
print("weighted AUC on the held-out MC (mass-blind raw inputs):")
print(f"  linear                       {H.auc(lin,  featfn, st['holdout']):.3f}")
print(f"  quadratic feature expansion  {H.auc(quad, featfn, st['holdout']):.3f}")
print(f"  3-layer neural net           {H.auc(mlp,  featfn, st['holdout']):.3f}")


### <font color="#8A4E17">What you found</font>
<hr color="#8A4E17">

Both models climb well clear of the linear score on the same raw inputs, now
with the capacity to build the mass. The net rediscovered, from four-momenta
alone, the variable physicists spent years learning to construct; that is the
result of Baldi, Sadowski and Whiteson (2014). Write down the AUC ordering you
got. Section 8 tests whether it means what it appears to mean.


## 4. The discovery test: a mass fit with a floating background

A classifier that separates simulated signal from simulated background is not
yet evidence for a particle. Section 1 left the data 18 percent above the
prediction, so a naive data-versus-simulation test would reject the null
because the normalization is wrong, not because a Higgs exists.

The fix is the generalized likelihood-ratio test from Lecture 5. Bin the events
in mass over 100-200 GeV and model the expected count in bin $i$ as

$$ \lambda_i = \mu\, S_i + \kappa\, B_i $$

where $S_i$ and $B_i$ are the signal and background templates from MC, $\mu$ is
the signal strength ($\mu=0$ means no Higgs, $\mu=1$ the Standard Model rate),
and $\kappa$ is a free background normalization, a nuisance parameter. Letting
$\kappa$ float absorbs an overall mis-normalization. We test $H_0:\mu=0$ with
$q = 2\log\frac{\max_{\mu,\kappa}L}{\max_\kappa L|_{\mu=0}}$.

Of the 495 events, 137 fall in the fit window. The classifier plays no part in
this section.

### <font color="#A6631F">Predict</font>
<hr color="#A6631F">

Given the excess of data over prediction and a possible peak near 125
GeV, where does $\hat\mu$ land, and how many sigma is the excess? Commit to
both numbers before running the fit.


## Your task 2: Fit the mass spectrum

Two functions. Both work on flat arrays of bin counts, so the same code
serves the 1D mass fit here and the 2D fit in Section 8.

First, the Poisson negative log-likelihood. For expected counts $\lambda_i$ and
observed counts $n_i$, dropping the constant $\log n_i!$ term,

$$ -\log L = \sum_i \big(\lambda_i - n_i \log \lambda_i\big) $$

Second, the test itself. Fit $\lambda_i = \mu S_i + \kappa B_i$ twice: once with
both parameters free, once with $\mu$ held at 0 and only $\kappa$ free. The
statistic is

$$ q = 2\big(\min_{\kappa} [-\log L]\big|_{\mu=0} - \min_{\mu,\kappa} [-\log L]\big) $$

floored at 0, and the asymptotic significance is $\sqrt q$. Use
`minimize(f, x0, bounds=...)` from `scipy.optimize`; the starting points and
bounds are given in the stub. Read $\kappa$ under the null off the second fit
and return it too, since Task 3 generates toys at that value.

In [ ]:
# STUDENT TASK 2: the binned Poisson likelihood and its GLRT.

def poisson_nll(lam, obs):
    """Negative log-likelihood of `obs` counts given expectations `lam`."""
    lam = np.maximum(np.asarray(lam, float).ravel(), 1e-9)
    obs = np.asarray(obs, float).ravel()
    # TODO 2a: return the sum over bins of (lam - obs * log(lam)).
    raise NotImplementedError("Return the Poisson NLL.")


def fit_glrt(S, B, obs):
    """Fit (mu, kappa) freely and under mu = 0; return both fits and q."""
    S = np.asarray(S, float).ravel()
    obs = np.asarray(obs, float).ravel()
    B = np.maximum(np.asarray(B, float).ravel(), 1e-9)

    # TODO 2b: two minimizations of poisson_nll over the model mu*S + kappa*B.
    #   free: parameters [mu, kappa], start [1.0, 1.2], bounds [(0, 20), (0.2, 5)]
    #   null: parameter  [kappa],     start [1.2],      bounds [(0.2, 5)]
    free = ...
    null = ...

    # TODO 2c: q = 2 * (null NLL - free NLL), never negative.
    q = ...

    return dict(mu_hat=free.x[0], kappa_hat=free.x[1], kappa_null=null.x[0],
                q=q, Z_asymptotic=np.sqrt(q))


In [ ]:
# Checkpoint 2: an analytic NLL, then the real fit against the reference.
assert np.isclose(poisson_nll([2.0], [3.0]), 2.0 - 3.0*np.log(2.0)), \
    "poisson_nll([2],[3]) should be 2 - 3*log(2)."
assert np.isclose(poisson_nll([1.0, 4.0], [0.0, 0.0]), 5.0), \
    "With no observed counts the NLL is just the sum of the expectations."

S, B, obs = H._templates(st)          # mass-only templates and observed counts
g = fit_glrt(S, B, obs)
_ref = H.glrt(S, B, obs)
for _k in ("mu_hat", "kappa_hat", "kappa_null", "q"):
    assert np.isclose(g[_k], _ref[_k], atol=1e-3), f"{_k} disagrees with the reference fit."

print(f"events in the 100-200 GeV fit window: {int(obs.sum())} of {len(data)}")
print(f"mu_hat    (signal strength)      = {g['mu_hat']:.3f}")
print(f"kappa_hat (background scale)      = {g['kappa_hat']:.3f}")
print(f"test statistic q                 = {g['q']:.3f}")
print(f"asymptotic significance sqrt(q)   = {g['Z_asymptotic']:.2f} sigma")
print("\u2713 Checkpoint 2 passed.")


In [ ]:
# Provided: your fitted model drawn over the observed mass spectrum.
edges = np.linspace(H.MASS_LO, H.MASS_HI, H.N_BINS + 1)
ctr = 0.5*(edges[:-1] + edges[1:])
plt.bar(ctr, obs, width=np.diff(edges), alpha=0.3, label="data", color="k")
plt.plot(ctr, g['kappa_hat']*B, "--", label=r"background only ($\hat\kappa\,B$)")
plt.plot(ctr, g['mu_hat']*S + g['kappa_hat']*B, "-", lw=2,
         label=r"signal + background ($\hat\mu S+\hat\kappa B$)")
plt.xlabel("four-lepton mass [GeV]"); plt.ylabel("events / bin"); plt.legend()
plt.title("GLRT mass fit: an excess near 125 GeV over a floating background")
plt.show()


The asymptotic $\sqrt{q}$ is an approximation, and $\mu = 0$ sits on the
boundary of the parameter space. The p-value that settles it comes from toys,
the Module 4 move: simulate background-only pseudo-datasets, refit each one,
and count how often $q$ reaches the observed value.


## Your task 3: Calibrate the p-value with toys

$\sqrt q$ assumes an asymptotic distribution that 137 events in 20 bins
may not have reached, and $\mu = 0$ sits on the boundary of the parameter space,
which breaks the standard argument. Get the null distribution by simulation
instead.

Generate `n_toys` background-only pseudo-datasets. Each toy draws an
independent Poisson count per bin with mean $\hat\kappa_0 B_i$, the background
fit under the null. Refit every toy with your `fit_glrt` and count how often its
$q$ reaches the observed $q$. Report

$$ p = \frac{\#\{q_\text{toy} \ge q_\text{obs}\} + 1}{n_\text{toys} + 1} $$

The $+1$ in both places keeps the estimate conservative and keeps $p$ away from
zero when no toy reaches the observation. Draw from `rng` in the loop so the
seed makes your run reproducible.

In [ ]:
# STUDENT TASK 3: the null distribution by simulation.

def toy_pvalue(S, B, obs, n_toys=2000, seed=SEED):
    """Background-only toys: p = P(q >= q_obs | mu = 0)."""
    fit = fit_glrt(S, B, obs)
    q_obs = fit["q"]
    B = np.maximum(np.asarray(B, float).ravel(), 1e-9)
    mean = fit["kappa_null"] * B          # expected counts under the fitted null
    rng = np.random.default_rng(seed)

    n_ge = 0
    for _ in range(n_toys):
        toy = ...        # TODO 3a: one Poisson pseudo-dataset with mean `mean`
        n_ge += ...      # TODO 3b: 1 if this toy's q reaches q_obs, else 0

    p = ...              # TODO 3c: the conservative (n_ge + 1) / (n_toys + 1)
    return dict(p_value=p, Z_toy=norm.isf(p), q_obs=q_obs)


In [ ]:
# Checkpoint 3: same seed, same toys, same answer as the reference.
_probe = toy_pvalue(S, B, obs, n_toys=200, seed=1)
_probe_ref = H.toy_pvalue(S, B, obs, n_toys=200, seed=1)
assert np.isclose(_probe["p_value"], _probe_ref["p_value"]), \
    "With the same seed your toys should reproduce the reference exactly."

tp = toy_pvalue(S, B, obs, n_toys=10000, seed=SEED)
print(f"toy-based p-value = {tp['p_value']:.4f}  (+/- {np.sqrt(tp['p_value']*(1-tp['p_value'])/10000):.4f} MC)")
print(f"significance      = {tp['Z_toy']:.2f} sigma  (one-sided)")
print("\u2713 Checkpoint 3 passed.")


## 5. How solid is this? Goodness-of-fit and systematics

A p-value is only as trustworthy as the model behind it. Our $\kappa$ absorbs
the background normalization, but it cannot absorb an error in the background
shape, so the significance above is conditional on the simulated mass shape
being right. Three checks follow: a goodness-of-fit test on the background-only
model, a refit with the shape freed by one tilt parameter, and confidence
intervals for $\mu$ under each assumption.

### <font color="#A6631F">Predict</font>
<hr color="#A6631F">

Which check hurts the result most? Write it down, then run.


## Your task 4: Test the fit itself

A p-value inherits the model behind it. Before trusting the excess, ask
whether the background-only model describes the spectrum at all.

Write the Poisson deviance goodness-of-fit test. For observed counts $n_i$ and
expected counts $e_i$,

$$ D = 2 \sum_i \Big[ n_i \log\frac{n_i}{e_i} - (n_i - e_i) \Big] $$

with the logarithmic term taken as 0 in any bin where $n_i = 0$. Under the
fitted model $D$ is approximately $\chi^2$ with (number of bins minus number of
fitted parameters) degrees of freedom, so the p-value is `chi2.sf(D, dof)`. A
small p-value here says the model misses the data, whatever the test of $\mu$
reported.

In [ ]:
# STUDENT TASK 4: Poisson deviance goodness-of-fit.

def deviance_gof(expected, obs, n_params):
    """Poisson deviance and its chi-square p-value."""
    e = np.maximum(np.asarray(expected, float).ravel(), 1e-9)
    o = np.asarray(obs, float).ravel()

    term = np.zeros_like(o)
    # TODO 4a: fill `term` with o*log(o/e) in the bins where o > 0, leaving 0
    #          elsewhere, then form the deviance D = 2 * sum(term - (o - e)).
    dev = ...

    dof = ...            # TODO 4b: bins minus fitted parameters
    p_value = ...        # TODO 4c: upper tail of the chi-square distribution
    return dict(deviance=float(dev), dof=int(dof), p_value=float(p_value))


In [ ]:
# Checkpoint 4: a hand-checkable case, then both fits from Section 4.
_d = deviance_gof([2.0, 2.0], [2.0, 2.0], n_params=1)
assert np.isclose(_d["deviance"], 0.0) and _d["dof"] == 1, \
    "A model matching the data exactly has zero deviance."
_z = deviance_gof([1.0], [0.0], n_params=0)
assert np.isclose(_z["deviance"], 2.0), \
    "With o = 0 the log term drops out and D = 2*(e - o)."

# (a) Does the background-only model even fit the data?
gof_null = deviance_gof(g["kappa_null"]*B, obs, n_params=1)
gof_sb   = deviance_gof(g["mu_hat"]*S + g["kappa_hat"]*B, obs, n_params=2)
assert np.isclose(gof_null["p_value"], H.goodness_of_fit(g["kappa_null"]*B, obs, 1)["p_value"], atol=1e-6)
print(f"background-only goodness-of-fit p = {gof_null['p_value']:.3f}")
print(f"signal+background   goodness-of-fit p = {gof_sb['p_value']:.3f}")
print("\u2713 Checkpoint 4 passed.")


In [ ]:
# Provided: let the background SHAPE flex by one tilt parameter, then re-test.
# lambda = mu*S + kappa*B*exp(t*x), with x the standardized bin center. The tilt
# is a generic flexibility check, not a model of any omitted process.
gs = H.glrt_shape(S, B, obs)
print(f"with a background-shape nuisance:  mu_hat = {gs['mu_hat']:.2f}, "
      f"significance = {gs['Z_asymptotic']:.2f} sigma  (was {g['Z_asymptotic']:.2f})")


## Your task 5: Put an interval on the signal strength

A significance says whether to reject $\mu = 0$. It does not say which
values of $\mu$ the data allow. Build the profile-likelihood interval.

For each candidate $\mu$, minimize the NLL over $\kappa$ with $\mu$ held fixed.
That gives the profile $-\log L_p(\mu)$. The $100\gamma$ percent interval is

$$ \Big\{\, \mu : 2\big[-\log L_p(\mu) + \log L_p(\hat\mu)\big] \le \chi^2_{1}(\gamma) \,\Big\} $$

Scan the given grid, keep the points that satisfy the inequality, and report the
smallest and largest. The critical value is `chi2.ppf(level, 1)`.

This interval holds the background shape fixed at the simulated prediction, so
it is conditional on that shape being right. The cell after it runs the matching
interval with the tilt free, and the two are worth reading side by side.

In [ ]:
# STUDENT TASK 5: profile-likelihood interval for mu.

def mu_interval(S, B, obs, level=0.95):
    """Profile-likelihood interval for mu, with kappa profiled out."""
    S = np.asarray(S, float).ravel()
    obs = np.asarray(obs, float).ravel()
    B = np.maximum(np.asarray(B, float).ravel(), 1e-9)

    def profile_nll(mu):
        # TODO 5a: minimize poisson_nll over kappa alone with mu held fixed
        #          (start [1.3], bounds [(0.2, 5)]) and return the minimized value.
        raise NotImplementedError("Return the profiled NLL at this mu.")

    mu_hat = fit_glrt(S, B, obs)["mu_hat"]
    fmin = profile_nll(mu_hat)
    thr = ...                              # TODO 5b: chi-square critical value, 1 dof
    grid = np.linspace(0.0, 5.0, 1000)
    inside = ...                           # TODO 5c: grid points inside the interval
    return dict(mu_hat=mu_hat, lo=float(inside.min()), hi=float(inside.max()), level=level)


In [ ]:
# Checkpoint 5: your interval against the reference, then the shape-flexed pair.
ci = mu_interval(S, B, obs)
_ref_ci = H.mu_confidence_interval(S, B, obs)
assert np.isclose(ci["lo"], _ref_ci["lo"], atol=1e-2) and np.isclose(ci["hi"], _ref_ci["hi"], atol=1e-2), \
    "Interval endpoints disagree with the reference."
assert ci["lo"] <= ci["mu_hat"] <= ci["hi"], "The interval must contain the point estimate."

# Provided: the same interval with kappa AND the shape tilt profiled out.
cis = H.mu_confidence_interval_shape(S, B, obs)
print(f"mu 95% CI, background shape fixed  = [{ci['lo']:.2f}, {ci['hi']:.2f}]")
print(f"mu 95% CI, background shape flexed = [{cis['lo']:.2f}, {cis['hi']:.2f}]")
print("\u2713 Checkpoint 5 passed.")


In [ ]:
# Provided: residual "pulls" per bin show where the background-only fit misses
pull = (obs - g["kappa_null"]*B) / np.sqrt(np.maximum(g["kappa_null"]*B, 1e-9))
plt.axhspan(-2, 2, color="0.85", label=r"$\pm2\sigma$")
plt.bar(ctr, pull, width=np.diff(edges), color="steelblue")
plt.axhline(0, color="k", lw=0.8)
plt.xlabel("four-lepton mass [GeV]"); plt.ylabel("(data - bkg) / sqrt(bkg)")
plt.title("Background-only pulls: the peak bins are high, but so is the scatter")
plt.legend(); plt.show()


### <font color="#8A4E17">What you found</font>
<hr color="#8A4E17">

The background-only fit is poor, and one tilt parameter takes a large bite out
of the significance. The intervals move with the same switch: with the shape
held fixed the 95 percent interval excludes zero, and with the shape free it
reaches zero and no longer does. The exclusion of $\mu = 0$ belongs to the
assumption, not to the data, so quote the fixed-shape interval only with that
label on it. And none of this yet counts
the two processes we never modeled (Section 1).

So which numbers do you report? Section 6 puts them together.


## 6. A hint, not a discovery

Your $\hat\mu$ should land near the Standard Model value of 1, but consistency
is cheap here: the fixed-shape interval runs from a small signal to well above
the SM rate, and the shape-free interval allows no signal at all. Compare your
two significances, one under the baseline model and one with the shape free.
Then check whether an ordinary $\alpha = 0.05$ decision even agrees between
them. Which background model you are willing to defend decides the verdict, and
that is with the particle-physics bars still far away: 3 sigma to call
something evidence, 5 sigma to call it a discovery.

Verdict: an excess worth watching. Not evidence, not a discovery. That is a
statement about how much information 500 events carry, and about how much of
the answer is coming from the simulation rather than the data.

History agrees. The 2012 announcement did not come from this channel alone;
each experiment reached 5 sigma by combining three decay channels, and the
four-lepton channel crossed 5 sigma on its own only with the full dataset,
roughly twice the sample you are holding. A hint here is what the record says
to expect.


## 7. Power: how much data would a discovery take?

This is the power calculation of Unit 2, run with the same GLRT. Hold the
physics at $\mu = 1$ and scale every expected count by $N$: the median
expected significance grows like $\sqrt N$, and the probability of actually
reaching 5 sigma is $\Phi(Z_\text{median} - 5)$. A median at 5 sigma is only
even odds.

Two assumptions ride along. The expected counts stand in for the median
experiment, an asymptotic step, and the background shape is held at the MC
prediction. Both flatter the projection: with the Section 5 tilt free, the same
luminosity multiple gives a visibly weaker median and power than the table below
will claim.

### <font color="#A6631F">Predict</font>
<hr color="#A6631F">

How many multiples of this dataset until the median crosses 5 sigma?
Write down your multiple, then run.


In [ ]:
# Provided: scale every expected count by N and re-run the same test.
proj = H.glrt_projection(st, factors=(1, 2, 4, 5, 6, 8, 10))
print(f"{'lumi':>5} {'median Z':>9} {'P(reach 5 sigma)':>18}")
for N, z_med, power in proj:
    print(f"{N:>4}x {z_med:9.2f} {power:18.2f}")

fac = [p[0] for p in proj]
fig, ax1 = plt.subplots()
ax1.plot(fac, [p[1] for p in proj], "o-", color="C0")
ax1.axhline(5, ls=":", color="r"); ax1.axhline(3, ls=":", color="gray")
ax1.set_xlabel("luminosity multiple"); ax1.set_ylabel("median expected Z [sigma]", color="C0")
ax2 = ax1.twinx()
ax2.plot(fac, [p[2] for p in proj], "s--", color="C3")
ax2.set_ylabel("power  P(reach 5 sigma)", color="C3"); ax2.set_ylim(0, 1)
plt.title("Median significance reaches 5 sigma near 5x; power lags behind"); plt.show()


### <font color="#8A4E17">What you found</font>
<hr color="#8A4E17">

Read off where the median crosses 5 sigma, then read the power in that same
row. The two do not arrive together, and a confident discovery in this channel
alone needs several times more data than the median crossing suggests, or a
better analysis, or the channel combination the LHC actually used.

Note what more data does and does not fix. It fixes the counting limit this
table measures. It does not fix a background shape that is subtly wrong, and
Section 5 showed one tilt parameter moving your significance a long way; that
error would survive a tenfold luminosity increase untouched. Scaling up an
analysis with a strained background model produces a more significant number,
not a more trustworthy one.

That is the case for more data. Section 8 asks whether a better statistic helps
instead.


## 8. What does the analysis gain from a classifier?

Section 3 left a question standing: the neural net won the AUC comparison by a
wide margin, so it should be the most useful model here. Is it?

Deciding that takes a metric that measures the right thing. Not how well a
score separates simulated signal from simulated background on its own, but how
much the full analysis improves when it uses the score. So we add the score to
the mass fit as a second dimension (a 2D mass-by-score GLRT, the way the LHC
experiments use classifier outputs) and read off the expected significance. It
runs on held-out MC only, never the 495 real events and never the rows a model
trained on, so nothing can rate rows it memorized.

The table also carries two reference rows, and they are the reason this section
is worth your attention. The 2D fit uses 20 mass bins times 4 score bins, so it
has 80 Poisson cells to work with, while the mass-only fit from Section 4 has
20. More cells buy resolution on their own, with no classifier involved, so
the second reference row splits the mass axis into 80 bins and uses no score at
all. The third row is a control: a score built only from the mass,
$h(x) = -|m - 125|$, carrying nothing the fit does not already have.

The third column is a diagnostic. Take the score cut that keeps 80 percent of
signal, and ask what fraction of the background *outside* the 115 to 135 GeV
window survives it. A score that has learned mass throws that background away,
which is what the collapse from 0.50 to 0.03 across the three models shows.
That matters because the sideband is what pins `kappa` down. A hard cut on a
mass-correlated score sculpts the background into the shape of a peak and
destroys the constraint the Section 4 fit relies on, which is why the score
enters as a fit dimension here and never as a cut.

### <font color="#A6631F">Predict</font>
<hr color="#A6631F">

Two predictions before you run the cell. Rank the three classifiers by
expected Z, and say whether any of them beats the 80-cell reference.


## Your task 6: Make the comparison fair

The 2D fit gets `H.N_BINS` mass bins times `H.N_SCORE_BINS` score bins.
The Section 4 mass-only fit gets `H.N_BINS`. Comparing them head to head credits
the classifier for every extra Poisson cell it was handed.

Set `N_REFERENCE_BINS` so the mass-only reference fits the same number of cells
as the 2D fit, using no score at all. The guard below refuses any other choice.
Then run the comparison and read the table against the prediction you wrote.

In [ ]:
# STUDENT TASK 6: choose the capacity-matched reference.
N_REFERENCE_BINS = 0        # <- your choice

_cells_2d = H.N_BINS * H.N_SCORE_BINS
if N_REFERENCE_BINS != _cells_2d:
    raise ValueError(
        f"A fair mass-only reference needs the same number of Poisson cells as "
        f"the 2D fit. The 2D fit uses {H.N_BINS} x {H.N_SCORE_BINS} cells."
    )
baseline20 = H.mass_only_expected_Z(st)                          # the Section 4 binning
baseline80 = H.mass_only_expected_Z(st, n_bins=N_REFERENCE_BINS)  # your matched reference
print(f"mass-only reference at {N_REFERENCE_BINS} cells: expected Z = {baseline80:.2f}")


In [ ]:
# Provided: the comparison table, built on the reference you just chose.
tag, tagfx = H.mass_tag_scorer()                     # control: mass and nothing else

print(f"{'reference':>26} {'AUC':>7} {'expected Z':>11} {'sideband ret.':>14}")
print(f"{'mass-only, 20 cells':>26} {'':>7} {baseline20:11.2f} {'':>14}")
print(f"{'mass-only, 80 cells':>26} {'':>7} {baseline80:11.2f} {'':>14}   <- fair reference")
r = H.comparison_row(tag, tagfx, st)
print(f"{'mass tag (no new info)':>26} {r['auc']:7.3f} {r['expected_Z']:11.2f} "
      f"{r['sideband_retention']:14.2f}")

print(f"\n{'classifier':>26} {'AUC':>7} {'expected Z':>11} {'sideband ret.':>14}")
for kind, (mdl, fx) in {"linear": (lin, featfn),
                        "quadratic feature exp.": (quad, featfn),
                        "3-layer neural net": (mlp, featfn)}.items():
    r = H.comparison_row(mdl, fx, st)
    print(f"{kind:>26} {r['auc']:7.3f} {r['expected_Z']:11.2f} "
          f"{r['sideband_retention']:14.2f}")


### <font color="#8A4E17">What you found</font>
<hr color="#8A4E17">

Read the table against both predictions.

Start with the classifier ordering by expected Z. It does not match the AUC
ordering from Section 3, so the first prediction has the answer that section set
up: the best separator is not the most useful model.

The second prediction is where the section earns its place. Compare every
classifier against your capacity-matched reference rather than against the
20-cell number, and the apparent gain does not survive. Then read the control
row: a score built from the mass and nothing else scores about as well as the
best classifier. A score that by construction knows nothing the fit did not
already know reproduces almost the entire apparent improvement.

The sideband column tells the same story from the other side: retention
collapses from 0.50 to 0.03 as the scores learn the mass.

So the improvement was never information. It was resolution. Splitting each
mass bin four ways sharpens the peak against the background, and any score
correlated with mass buys that sharpening whether or not it contributes
anything new. The 20-cell comparison handed the classifiers credit for degrees
of freedom rather than for physics.

This is the trap worth carrying out of the project. A more elaborate model beat
a baseline, the margin looked real, and it dissolved once the baseline was
given the same capacity. Before believing that a model improved on a simpler
one, check what else changed alongside it: bins, parameters, features, tuning
budget. The comparison has to hold those fixed or the number it produces is
measuring the wrong thing.

Two loose ends worth naming rather than hiding. Why the net lands below the
quadratic is not settled here, and one candidate is visible in the code rather
than the physics. `sklearn`'s MLP takes no sample weights, so it is
class-balanced by resampling instead: 25,000 draws per class, which is at most
25,000 of the 62,382 unique background rows, while the other two models see all
of them. The three models are therefore not matched on training data, which is
the same kind of unexamined difference this section was written to warn about.
Naming it is the honest move; measuring it is a separate experiment. And a real
gain would need discrimination decorrelated from mass, which in this channel
means reconstructing the Z pair, particle physics beyond this course. Building
models that are powerful and decorrelated at once is where the deep learning
course picks up.

## 9. What you can, and cannot, claim

Be precise about what the test established. The GLRT compared the data against
a specific background-only model and rejected it at a stated level. That is a
statement about $P_0$: the data are not well described by the simulated
background alone. It is not a statement that the data follow
$\mu S + \kappa B$ with $\mu > 0$.

Which background-only model matters more than it sounds, because this project
fit two of them and your own numbers should show them disagreeing at the
conventional 5 percent level. The fixed-shape baseline rejects. The
shape-flexible version, the same background with one tilt free, does not. A
rejection is always a rejection of a specified null, and here the specification
is doing much of the work.

Rejecting the null also does not single out the alternative you had in mind. A
small excess near 125 GeV is consistent with a Standard Model Higgs, but the
same rejection could come from a mismodeled simulation (Section 5 showed this
is live), from a process the simulation omits (Section 1 named two), or from
an ordinary fluctuation of a background behaving as simulated. On the last
one, mind the conditional: if $H_0$ is true, rejecting it is a Type I error,
and a p-value of one in a hundred means that happens about once per hundred
such analyses. Whether this rejection is one depends on whether $H_0$ is true,
which is what we do not know.

The floating $\kappa$ protects against an overall normalization error and
certifies nothing else. This is why particle physics does not stop at "the
null is rejected." The 5 sigma bar and the demand for corroboration across
channels and experiments are conventions built to keep lucky fluctuations from
becoming particles. They are strong norms rather than laws of inference, and
they buy no protection against a background model that is simply wrong: a
sufficiently wrong simulation produces a very significant wrong answer. The
remedy for that is a better model and independent corroboration, not a smaller
p-value.

Task 7 asks you to state, in these terms, what this result does and does not
license. The graded checkpoint follows it. Work through both before entering
your numbers below.


## Your task 7: State what you can claim

Write the record the analysis supports. Four fields, each a specific
statement rather than a summary of the section.

- `CLAIM`: what your test established, in terms of the model it rejected. Be
  careful to distinguish a statement about the background-only model from a
  statement about the signal strength.
- `EVIDENCE`: at least one number you computed, with the assumption it depends
  on named alongside it.
- `CAVEAT`: the assumption that would change the verdict if it were wrong, and
  which direction it would move.
- `NEXT_EXPERIMENT`: one experiment that changes a single factor, and what
  result would settle the question.

The contract checks that the record is complete and specific. It does not check
that you reached a prescribed verdict.

In [ ]:
# STUDENT TASK 7: replace each empty string with your own record.
CLAIM = ""
EVIDENCE = ""
CAVEAT = ""
NEXT_EXPERIMENT = ""

learner_record = {
    "claim": CLAIM.strip(),
    "evidence": EVIDENCE.strip(),
    "caveat": CAVEAT.strip(),
    "next_experiment": NEXT_EXPERIMENT.strip(),
}
if any(len(learner_record[f]) < 40 for f in learner_record):
    raise ValueError("Each field needs a specific statement of at least 40 characters.")
if not any(ch.isdigit() for ch in learner_record["evidence"]):
    raise ValueError("Cite at least one number you computed in EVIDENCE.")
for _f, _t in learner_record.items():
    print(f"{_f}:\n  {_t}\n")
print("\u2713 Record captured.")


## 10. Report your results

Enter these on the submission page. They come from the frozen pipeline above.


In [ ]:
# Provided: assembled from your fit, your toys, your goodness-of-fit, and your
# interval. Run every cell above in order first.
results = {
    "mu_hat":                 round(g["mu_hat"], 3),
    "kappa_hat":              round(g["kappa_hat"], 3),
    "glrt_q":                 round(g["q"], 3),
    "toy_p_value":            round(tp["p_value"], 4),
    "significance_baseline":  round(tp["Z_toy"], 2),
    "significance_with_shape":round(gs["Z_asymptotic"], 2),
    "mu_CI_low":              round(ci["lo"], 2),      # background shape FIXED
    "mu_CI_high":             round(ci["hi"], 2),      # background shape FIXED
    "mu_CI_shape_low":        round(cis["lo"], 2),     # background shape FLEXED
    "mu_CI_shape_high":       round(cis["hi"], 2),     # background shape FLEXED
    "background_gof_p":       round(gof_null["p_value"], 3),
    "auc_quadratic":          round(H.auc(quad, featfn, st["holdout"]), 3),
    "baseline_expected_Z":    round(baseline20, 2),
    "matched_baseline_Z":     round(baseline80, 2),
    "lumi_factor_median_5sigma": next(N for N, z, _ in
                                      H.glrt_projection(st, factors=range(1, 13)) if z >= 5),
    "power_at_5x": round(next(p for N, z, p in
                              H.glrt_projection(st, factors=[5])), 2),
}
for k, v in results.items():
    print(f"{k:28s} = {v}")

print("\nmu_CI_high is the fixed-shape bound; with the shape free, the interval reaches mu = 0.")


## Pause and reflect (ungraded)

**Q1.** Your `fit_glrt` floors $q$ at zero. What would a negative $q$ mean, and
why can it happen numerically when it cannot happen mathematically?

**Q2.** Task 3 adds 1 to both the numerator and the denominator of the
p-value. What goes wrong at 10,000 toys if you do not?

**Q3.** The goodness-of-fit test and the test of $\mu = 0$ can disagree: a
model can fit badly and still show a significant excess, or fit well and show
none. What does each one license you to say?

**Q4.** In Task 5, the fixed-shape interval and the shape-flexed interval come
from the same data. Explain why the second is wider without using the word
uncertainty.

**Q5.** Task 6 forced the reference to match the 2D fit on cell count. Name one
other thing you would have to hold fixed before crediting a model with an
improvement.

**Q6.** Section 3 ranked models by AUC and Section 8 ranked them by expected
significance. Which quantity would you report to a physicist deciding whether to
adopt your classifier, and why?

**Q7.** Task 1 balanced the classes. Suppose you had trained on the physical
weights instead. Where would a threshold at $z = 0$ sit on the
likelihood-ratio scale, and would a test using that threshold still be valid?